# Auditoría de discrepancias COT–OC — Agosto 2026

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/MercadoPublico-Inteligencia/blob/main/colab/Auditoria_Discrepancias_COT_OC_Colab.ipynb)

Análisis reproducible **sin escritura en Google Sheets ni llamadas a API**. Archivos de entrada: CSV de detalle y conciliación por proceso de la segunda auditoría. Clasifica similitud textual de proveedores y diferencias de monto. **La similitud de nombres no verifica identidad jurídica; se requiere RUT.**

In [ ]:
!pip -q install pandas
import pandas as pd,re,unicodedata,json,collections
from difflib import SequenceMatcher
from decimal import Decimal, InvalidOperation
from google.colab import files

print("Sube auditoria_cot_oc_corregida_2026_08_detalle.csv y conciliacion_COT_OC_agosto_2026_procesos.csv")
subidos=files.upload()
d=pd.read_csv("auditoria_cot_oc_corregida_2026_08_detalle.csv",dtype=str,keep_default_na=False)
co=pd.read_csv("conciliacion_COT_OC_agosto_2026_procesos.csv",dtype=str,keep_default_na=False)
assert len(d)==68093 and len(co)==65945, "Volumen diferente del corte documentado: revisar antes de seguir"

def norm(s):
 s=unicodedata.normalize("NFKD",s.upper())
 s="".join(c for c in s if not unicodedata.combining(c))
 return re.sub("[^A-Z0-9]","",s)
def tokens(s):
 s=unicodedata.normalize("NFKD",s.upper())
 s="".join(c for c in s if not unicodedata.combining(c))
 s=re.sub("[^A-Z0-9]+"," ",s)
 stop={"SOCIEDAD","DE","LA","EL","Y","SPA","S","A","LTDA","LIMITADA","EIRL","EMPRESA","COMERCIAL","INVERSIONES"}
 return set(s.split())-stop
D=d[d.categoria=="CODIGO_OC_COINCIDE_PROVEEDOR_DISTINTO"].copy()
similitudes=[];solapamientos=[];prioridades=[]
for a,b in zip(D.proveedor_seleccionado,D.proveedor_oc):
 sim=SequenceMatcher(None,norm(a),norm(b)).ratio()
 A,B=tokens(a),tokens(b)
 sol=len(A&B)/len(A|B) if A|B else 0
 similitudes.append(round(sim,3));solapamientos.append(round(sol,3))
 prioridades.append("SIMILITUD_ALTA_REVISAR" if sim>=.85 or sol>=.8 else
                    ("SIMILITUD_MEDIA_REVISAR" if sim>=.65 or sol>=.5 else "NOMBRES_DISIMILES_VERIFICAR_RUT"))
D["similitud_textual"]=similitudes
D["solapamiento_tokens"]=solapamientos
D["prioridad_revision"]=prioridades
M=d[d.categoria=="CODIGO_OC_PROVEEDOR_COINCIDEN_MONTO_DIFERENTE"].copy()
def dec(s):
 try:return Decimal(s.replace(",","."))
 except InvalidOperation:return None
M["oferta_no_entera"]=M.monto_oferta_clp.map(lambda s:dec(s) is None or dec(s)!=dec(s).to_integral_value())
M["diferencia_absoluta_clp"]=M.apply(lambda r:str(abs(dec(r.monto_oc_clp)-dec(r.monto_oferta_clp)))
 if dec(r.monto_oc_clp) is not None and dec(r.monto_oferta_clp) is not None else "",axis=1)
D.to_csv("auditoria_discrepancias_proveedores_agosto.csv",index=False,encoding="utf-8-sig")
M.to_csv("auditoria_discrepancias_montos_agosto.csv",index=False,encoding="utf-8-sig")
resumen={"fuente_detalle_filas":len(d),"fuente_conciliacion_procesos":len(co),
 "diferencias_proveedor_vinculos":len(D),"diferencias_proveedor_cot_distintas":int(D.codigo_cot.nunique()),
 "niveles_similitud":dict(collections.Counter(prioridades)),
 "diferencias_monto_vinculos":len(M),"diferencias_monto_cot_distintas":int(M.codigo_cot.nunique()),
 "montos_oferta_no_enteros":int(M.oferta_no_entera.sum()),
 "nota":"Similitud textual no acredita identidad. Solo lectura."}
with open("auditoria_discrepancias_resumen.json","w") as f:json.dump(resumen,f,ensure_ascii=False,indent=2)
print(json.dumps(resumen,ensure_ascii=False,indent=2))
for p in ["auditoria_discrepancias_proveedores_agosto.csv","auditoria_discrepancias_montos_agosto.csv","auditoria_discrepancias_resumen.json"]:files.download(p)